In [1]:
import sys
!{sys.executable} -m pip install -q torch_geometric



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path

PROJECT = Path("/home/ibm-user/shared")
DATA_DIR = PROJECT / "data"
RESULTS_DIR = PROJECT / "results"
SCORES_DIR = RESULTS_DIR / "scores"

SCORES_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Dataset exists:", (DATA_DIR / "HI-Small_Trans.csv").exists())
print("Scores directory:", SCORES_DIR)
print("Processed data exists:",
      (DATA_DIR / "processed" / "nodes.csv").exists())

Project: /home/ibm-user/shared
Dataset exists: True
Scores directory: /home/ibm-user/shared/results/scores
Processed data exists: False


In [3]:
import importlib

packages = [
    "torch",
    "torch_geometric",
    "sklearn",
    "xgboost",
    "matplotlib",
    "networkx",
]

for package in packages:
    try:
        mod = importlib.import_module(package)
        print(f"{package}: OK",
              getattr(mod, "__version__", ""))
    except Exception as exc:
        print(f"{package}: ERROR - {exc}")

torch: OK 2.11.0+cpu
torch_geometric: OK 2.8.1
sklearn: OK 1.9.0
xgboost: OK 3.4.1
matplotlib: OK 3.11.2
networkx: OK 3.7


In [4]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)


def evaluate_scores(y_true, scores, alert_fraction=0.01):
    """
    Evaluate account risk scores at a fixed alert budget.

    The caller must supply scores for the intended evaluation
    population only, with labels from the fixed test split.
    """
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)

    if len(y) != len(s) or len(y) == 0:
        raise ValueError("Labels and scores must have equal, nonzero length.")

    if not np.isin(y, [0, 1]).all():
        raise ValueError("Labels must be binary: 0 or 1.")

    if not np.isfinite(s).all():
        raise ValueError("Scores contain NaN or infinite values.")

    if y.sum() == 0 or y.sum() == len(y):
        raise ValueError("Both positive and negative labels are required.")

    if not 0 < alert_fraction <= 1:
        raise ValueError("alert_fraction must be in (0, 1].")

    # Select the highest-risk accounts.
    # Stable sorting makes ties reproducible.
    n_alerts = max(1, int(np.ceil(len(y) * alert_fraction)))
    order = np.argsort(-s, kind="stable")
    alerted = np.zeros(len(y), dtype=bool)
    alerted[order[:n_alerts]] = True

    return {
        "n_test": len(y),
        "n_mules": int(y.sum()),
        "n_alerts": int(alerted.sum()),
        "PR_AUC": float(average_precision_score(y, s)),
        "ROC_AUC": float(roc_auc_score(y, s)),
        "precision_at_budget": float(
            precision_score(y, alerted, zero_division=0)
        ),
        "recall_at_budget": float(
            recall_score(y, alerted, zero_division=0)
        ),
        "F1_at_budget": float(
            f1_score(y, alerted, zero_division=0)
        ),
    }


print("Evaluation function defined successfully.")

Evaluation function defined successfully.


In [5]:
# Small artificial test to validate the metric implementation.
# These values are only a software test, NOT project results.

y_test = np.array([0, 1, 0, 1, 0, 0, 1, 0, 0, 1])
scores_test = np.array([
    0.10, 0.95, 0.20, 0.80, 0.30,
    0.40, 0.70, 0.15, 0.25, 0.60
])

test_metrics = evaluate_scores(
    y_test,
    scores_test,
    alert_fraction=0.20
)

print(pd.Series(test_metrics).to_string())

n_test                 10.000000
n_mules                 4.000000
n_alerts                2.000000
PR_AUC                  1.000000
ROC_AUC                 1.000000
precision_at_budget     1.000000
recall_at_budget        0.500000
F1_at_budget            0.666667


In [6]:
def evaluate_by_member(
    predictions,
    labels,
    alert_fraction=0.01
):
    """
    predictions columns:
        acct_key, inst, split, score

    labels columns:
        acct_key, inst, split, label

    Evaluates test accounts only.
    Applies the alert budget separately for each member.
    """

    required_pred = {"acct_key", "inst", "split", "score"}
    required_labels = {"acct_key", "inst", "split", "label"}

    if not required_pred.issubset(predictions.columns):
        raise ValueError(
            f"Missing prediction columns: "
            f"{required_pred - set(predictions.columns)}"
        )

    if not required_labels.issubset(labels.columns):
        raise ValueError(
            f"Missing label columns: "
            f"{required_labels - set(labels.columns)}"
        )

    pred = predictions[predictions["split"] == "test"].copy()
    lab = labels[["acct_key", "inst", "split", "label"]].copy()
    lab = lab[lab["split"] == "test"]

    if pred["acct_key"].duplicated().any():
        raise ValueError("Duplicate account IDs in predictions.")

    if lab["acct_key"].duplicated().any():
        raise ValueError("Duplicate account IDs in labels.")

    merged = pred.merge(
        lab,
        on=["acct_key", "inst", "split"],
        how="left",
        validate="one_to_one",
        indicator=True
    )

    if not (merged["_merge"] == "both").all():
        raise ValueError(
            "Some prediction accounts have no matching test labels."
        )

    merged = merged.drop(columns="_merge")

    if merged.empty:
        raise ValueError("No matching test predictions found.")

    results = []

    for member, group in merged.groupby("inst", sort=True):
        if group["label"].nunique() != 2:
            print(f"Skipping {member}: test set needs both classes.")
            continue

        metrics = evaluate_scores(
            group["label"].to_numpy(),
            group["score"].to_numpy(),
            alert_fraction=alert_fraction
        )

        results.append({
            "inst": member,
            **metrics
        })

    if not results:
        raise ValueError("No members could be evaluated.")

    return pd.DataFrame(results)


print("Per-member evaluator defined successfully.")

Per-member evaluator defined successfully.


In [7]:
def compare_recall(silo_results, combined_results):
    """
    Compare per-member recall at the same alert budget.
    Both inputs must come from the same test-account definitions.
    """

    comparison = silo_results[
        ["inst", "recall_at_budget"]
    ].merge(
        combined_results[["inst", "recall_at_budget"]],
        on="inst",
        how="inner",
        suffixes=("_silo", "_combined"),
        validate="one_to_one"
    )

    comparison["uplift_percentage_points"] = (
        comparison["recall_at_budget_combined"]
        - comparison["recall_at_budget_silo"]
    ) * 100

    comparison["relative_uplift_percent"] = np.where(
        comparison["recall_at_budget_silo"] > 0,
        (
            comparison["recall_at_budget_combined"]
            / comparison["recall_at_budget_silo"] - 1
        ) * 100,
        np.nan
    )

    return comparison


print("Recall comparison function defined successfully.")

Recall comparison function defined successfully.


In [8]:
from pathlib import Path

for relative_path in [
    "data/processed/nodes.csv",
    "data/processed/edges.npz",
    "mulenet/common.py",
    "results/scores/lr_silo.csv",
    "results/scores/lr_combined.csv",
    "results/scores/gnn_silo.csv",
    "results/scores/gnn_combined.csv",
]:
    path = PROJECT / relative_path
    print(f"{'READY' if path.exists() else 'WAITING'}  {relative_path}")

WAITING  data/processed/nodes.csv
WAITING  data/processed/edges.npz
WAITING  mulenet/common.py
WAITING  results/scores/lr_silo.csv
WAITING  results/scores/lr_combined.csv
WAITING  results/scores/gnn_silo.csv
WAITING  results/scores/gnn_combined.csv


In [9]:
# Software test only — NOT actual MuleNet-Z results.

demo_nodes = pd.DataFrame({
    "acct_key": [f"{bank}_{i}" for bank in ["B1", "B2"]
                 for i in range(100)],
    "inst": [bank for bank in ["B1", "B2"] for _ in range(100)],
    "split": ["test"] * 200,
    "label": [int(i < 10) for _ in ["B1", "B2"] for i in range(100)],
})

demo_silo = demo_nodes[["acct_key", "inst", "split"]].copy()
demo_silo["score"] = demo_nodes["label"] * 0.5 + np.linspace(
    0, 0.4, len(demo_nodes)
)

demo_combined = demo_silo.copy()
demo_combined["score"] = np.where(
    demo_nodes["label"].to_numpy() == 1,
    0.99,
    np.linspace(0.01, 0.50, len(demo_nodes))
)

silo_metrics = evaluate_by_member(
    demo_silo, demo_nodes, alert_fraction=0.10
)
combined_metrics = evaluate_by_member(
    demo_combined, demo_nodes, alert_fraction=0.10
)

print("Silo test:")
display(silo_metrics)

print("Combined test:")
display(combined_metrics)

print("Recall comparison:")
display(compare_recall(silo_metrics, combined_metrics))

Silo test:


,inst,n_test,n_mules,n_alerts,PR_AUC,ROC_AUC,precision_at_budget,recall_at_budget,F1_at_budget
0,B1,100,10,10,1.0,1.0,1.0,1.0,1.0
1,B2,100,10,10,1.0,1.0,1.0,1.0,1.0


Combined test:


,inst,n_test,n_mules,n_alerts,PR_AUC,ROC_AUC,precision_at_budget,recall_at_budget,F1_at_budget
0,B1,100,10,10,1.0,1.0,1.0,1.0,1.0
1,B2,100,10,10,1.0,1.0,1.0,1.0,1.0


Recall comparison:


,inst,recall_at_budget_silo,recall_at_budget_combined,uplift_percentage_points,relative_uplift_percent
0,B1,1.0,1.0,0.0,0.0
1,B2,1.0,1.0,0.0,0.0


In [10]:
from pathlib import Path

for folder_name in ["scores", "scores_sample"]:
    folder = PROJECT / "results" / folder_name
    print(f"\n{folder_name}:")

    if not folder.exists():
        print("  Directory not created yet.")
        continue

    files = sorted(folder.glob("*.csv"))
    if not files:
        print("  No score CSVs yet.")
    else:
        for file in files:
            print(f"  {file.name} — {file.stat().st_size:,} bytes")


scores:
  No score CSVs yet.

scores_sample:
  Directory not created yet.


In [11]:
import importlib.util

packages = ["pandas", "numpy", "sklearn", "xgboost"]

for package in packages:
    spec = importlib.util.find_spec(package)
    print(f"{package}: {'AVAILABLE' if spec else 'NOT INSTALLED'}")

pandas: AVAILABLE
numpy: AVAILABLE
sklearn: AVAILABLE
xgboost: AVAILABLE


In [12]:
import csv

dataset_path = PROJECT / "data" / "HI-Small_Trans.csv"

with open(dataset_path, "r", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    columns = next(reader)
    sample_rows = [next(reader) for _ in range(3)]

print("Columns:")
for column in columns:
    print("-", column)

print("\nFirst 3 data rows:")
for row in sample_rows:
    print(row)

Columns:
- Timestamp
- From Bank
- Account
- To Bank
- Account
- Amount Received
- Receiving Currency
- Amount Paid
- Payment Currency
- Payment Format
- Is Laundering

First 3 data rows:
['2022/09/01 00:20', '010', '8000EBD30', '010', '8000EBD30', '3697.34', 'US Dollar', '3697.34', 'US Dollar', 'Reinvestment', '0']
['2022/09/01 00:20', '03208', '8000F4580', '001', '8000F5340', '0.01', 'US Dollar', '0.01', 'US Dollar', 'Cheque', '0']
['2022/09/01 00:00', '03209', '8000F4670', '03209', '8000F4670', '14675.57', 'US Dollar', '14675.57', 'US Dollar', 'Reinvestment', '0']


In [16]:
import pandas as pd
from collections import Counter

dataset_path = PROJECT / "data" / "HI-Small_Trans.csv"

rows_scanned = 0
label_counts = Counter()
sender_banks = set()
receiver_banks = set()
payment_formats = Counter()
currencies = Counter()
missing_by_column = Counter()
invalid_amount_rows = 0
min_timestamp = None
max_timestamp = None

for chunk in pd.read_csv(dataset_path, chunksize=100_000):
    rows_scanned += len(chunk)

    # The CSV contains two columns named "Account".
    # Pandas normally renames the second one to "Account.1".
    label_col = "Is Laundering"

    label_counts.update(
        chunk[label_col].astype(str).str.strip()
    )

    sender_banks.update(
        chunk["From Bank"].dropna().astype(str).unique()
    )
    receiver_banks.update(
        chunk["To Bank"].dropna().astype(str).unique()
    )

    payment_formats.update(
        chunk["Payment Format"].dropna().astype(str)
    )

    currencies.update(
        chunk["Payment Currency"].dropna().astype(str)
    )

    missing_by_column.update(
        chunk.isna().sum().to_dict()
    )

    for col in ["Amount Received", "Amount Paid"]:
        values = pd.to_numeric(chunk[col], errors="coerce")
        invalid_amount_rows += int(values.isna().sum())

    timestamps = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="coerce"
    ).dropna()

    if not timestamps.empty:
        chunk_min = timestamps.min()
        chunk_max = timestamps.max()

        min_timestamp = (
            chunk_min if min_timestamp is None
            else min(min_timestamp, chunk_min)
        )
        max_timestamp = (
            chunk_max if max_timestamp is None
            else max(max_timestamp, chunk_max)
        )

print("Rows scanned:", f"{rows_scanned:,}")
print("Transaction labels:", dict(label_counts))
print("Distinct sender banks:", len(sender_banks))
print("Distinct receiver banks:", len(receiver_banks))
print("Payment formats:", dict(payment_formats))
print("Number of payment currencies:", len(currencies))
print("Timestamp range:", min_timestamp, "to", max_timestamp)
print("Missing values:", dict(missing_by_column))
print("Invalid amount values across both amount columns:",
      invalid_amount_rows)


Rows scanned: 5,078,345
Transaction labels: {'0': 5073168, '1': 5177}
Distinct sender banks: 30470
Distinct receiver banks: 15811
Payment formats: {'Reinvestment': 481056, 'Cheque': 1864331, 'Credit Card': 1323324, 'ACH': 600797, 'Cash': 490891, 'Wire': 171855, 'Bitcoin': 146091}
Number of payment currencies: 15
Timestamp range: 2022-09-01 00:00:00 to 2022-09-18 16:18:00
Missing values: {'Timestamp': 0, 'From Bank': 0, 'Account': 0, 'To Bank': 0, 'Account.1': 0, 'Amount Received': 0, 'Receiving Currency': 0, 'Amount Paid': 0, 'Payment Currency': 0, 'Payment Format': 0, 'Is Laundering': 0}
Invalid amount values across both amount columns: 0


In [ ]:
from pathlib import Path
import pandas as pd

SCORE_DIRS = [
    PROJECT / "results" / "scores_sample",
    PROJECT / "results" / "scores",
]

REQUIRED_COLUMNS = {"acct_key", "inst", "split", "score"}

for folder in SCORE_DIRS:
    print(f"\nChecking: {folder}")

    if not folder.is_dir():
        print("  Not available yet.")
        continue

    files = sorted(folder.glob("gnn_*.csv"))

    if not files:
        print("  No GNN score files yet.")
        continue

    for file in files:
        try:
            scores = pd.read_csv(file)

            missing = REQUIRED_COLUMNS - set(scores.columns)

            if missing:
                print(f"  INVALID {file.name}: missing {sorted(missing)}")
                continue

            print(
                f"  OK {file.name}: "
                f"{len(scores):,} rows, "
                f"{scores['acct_key'].nunique():,} unique accounts, "
                f"splits={scores['split'].value_counts().to_dict()}"
            )

        except Exception as exc:
            print(f"  ERROR {file.name}: {exc}")

In [19]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

PROJECT = Path("/home/ibm-user/shared")
RESULTS_DIR = PROJECT / "results" / "niyati"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

for path in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

ALERT_BUDGET = 0.01

print("Project:", PROJECT)
print("Your results directory:", RESULTS_DIR)
print("Alert budget:", ALERT_BUDGET)

Project: /home/ibm-user/shared
Your results directory: /home/ibm-user/shared/results/niyati
Alert budget: 0.01


In [21]:
def validate_scores(scores):
    required = {"acct_key", "inst", "split", "score"}
    missing = required - set(scores.columns)

    if missing:
        raise ValueError(f"Missing prediction columns: {sorted(missing)}")

    df = scores.copy()

    if df[["acct_key", "inst", "split", "score"]].isna().any().any():
        raise ValueError("Predictions contain missing required values.")

    df["acct_key"] = df["acct_key"].astype(str)
    df["inst"] = df["inst"].astype(str)
    df["split"] = df["split"].astype(str)
    df["score"] = pd.to_numeric(df["score"], errors="raise")

    if not np.isfinite(df["score"]).all():
        raise ValueError("Scores contain NaN or infinite values.")

    if df.duplicated(["acct_key", "inst", "split"]).any():
        raise ValueError("Duplicate account predictions detected.")

    if not df["split"].isin(["train", "val", "test"]).all():
        raise ValueError("Unexpected split value.")

    return df


def validate_labels(labels):
    required = {"acct_key", "inst", "split", "label"}
    missing = required - set(labels.columns)

    if missing:
        raise ValueError(f"Missing label columns: {sorted(missing)}")

    df = labels.copy()
    df["acct_key"] = df["acct_key"].astype(str)
    df["inst"] = df["inst"].astype(str)
    df["split"] = df["split"].astype(str)

    if df[["acct_key", "inst", "split", "label"]].isna().any().any():
        raise ValueError("Labels contain missing required values.")

    if not df["label"].isin([0, 1]).all():
        raise ValueError("Labels must be binary: 0 or 1.")

    if df.duplicated(["acct_key", "inst", "split"]).any():
        raise ValueError("Duplicate account labels detected.")

    return df

In [22]:
def evaluate_model(scores, labels, budget=ALERT_BUDGET):
    scores = validate_scores(scores)
    labels = validate_labels(labels)

    if not 0 < budget <= 1:
        raise ValueError("Budget must be greater than 0 and at most 1.")

    # Only score the held-out test accounts.
    pred = scores[scores["split"] == "test"].copy()
    truth = labels[labels["split"] == "test"].copy()

    # Require an exact one-to-one match between scored and labeled accounts.
    keys = ["acct_key", "inst", "split"]

    merged = pred.merge(
        truth[keys + ["label"]],
        on=keys,
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    if not (merged["_merge"] == "both").all():
        counts = merged["_merge"].value_counts().to_dict()
        raise ValueError(f"Prediction/label account mismatch: {counts}")

    merged = merged.drop(columns="_merge")
    merged["label"] = merged["label"].astype(int)

    rows = []

    for inst, group in merged.groupby("inst", sort=True):
        y = group["label"].to_numpy()
        s = group["score"].to_numpy()

        # Keep this rule consistent with the GNN evaluator.
        k = max(1, int(round(budget * len(group))))
        top = group.nlargest(k, "score")
        predicted = group.index.isin(top.index)

        precision = precision_score(
            y, predicted, zero_division=0
        )
        recall = recall_score(
            y, predicted, zero_division=0
        )
        f1 = f1_score(
            y, predicted, zero_division=0
        )

        rows.append({
            "inst": inst,
            "test_accounts": len(group),
            "positive_accounts": int(y.sum()),
            "alerts": k,
            "caught": int(top["label"].sum()),
            "recall_at_budget": recall,
            "precision_at_budget": precision,
            "f1_at_budget": f1,
            "pr_auc": (
                average_precision_score(y, s)
                if len(np.unique(y)) == 2 else np.nan
            ),
            "roc_auc": (
                roc_auc_score(y, s)
                if len(np.unique(y)) == 2 else np.nan
            ),
        })

    return pd.DataFrame(rows)

In [23]:
def summarize_model(model_name, metrics):
    total_positives = metrics["positive_accounts"].sum()
    total_caught = metrics["caught"].sum()

    return {
        "model": model_name,
        "institutions": metrics["inst"].nunique(),
        "test_accounts": metrics["test_accounts"].sum(),
        "alerts": metrics["alerts"].sum(),
        "positive_accounts": total_positives,
        "caught": total_caught,
        "pooled_recall_at_budget": (
            total_caught / total_positives
            if total_positives else np.nan
        ),
        "macro_recall_at_budget": metrics["recall_at_budget"].mean(),
        "macro_precision_at_budget": metrics["precision_at_budget"].mean(),
        "macro_f1_at_budget": metrics["f1_at_budget"].mean(),
        "macro_pr_auc": metrics["pr_auc"].mean(),
        "macro_roc_auc": metrics["roc_auc"].mean(),
    }


def build_comparison(model_results):
    """
    model_results: dict of model name -> institution-level metrics DataFrame
    """
    rows = []

    for model_name, metrics in model_results.items():
        rows.append(summarize_model(model_name, metrics))

    return pd.DataFrame(rows)

In [24]:
import matplotlib.pyplot as plt


def plot_model_comparison(comparison, metric, title, output_name):
    if metric not in comparison.columns:
        raise ValueError(f"Metric not found: {metric}")

    data = comparison[["model", metric]].dropna()

    if data.empty:
        print(f"No real values available yet for {metric}.")
        return

    ax = data.plot(
        x="model",
        y=metric,
        kind="line",
        marker="o",
        legend=False,
        figsize=(10, 5),
    )

    ax.set_title(title)
    ax.set_xlabel("Model")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = FIGURES_DIR / output_name
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()

    print("Saved:", path)

In [27]:
# Use these after real results have been loaded.
plot_model_comparison(
    comparison,
    "pooled_recall_at_budget",
    "Recall at 1% Alert Budget",
    "recall_at_budget.png",
)

plot_model_comparison(
    comparison,
    "macro_pr_auc",
    "Macro PR-AUC by Model",
    "macro_pr_auc.png",
)

NameError: name 'comparison' is not defined

In [26]:
manifest = {
    "project": "MuleNet-Z",
    "role": "Role 3 - Baselines and Evaluation",
    "alert_budget": ALERT_BUDGET,
    "alert_budget_scope": "per institution",
    "alert_count_rule": "max(1, round(budget * test_accounts_per_institution))",
    "primary_metric": "recall_at_budget",
    "supporting_metrics": [
        "pr_auc",
        "roc_auc",
        "precision_at_budget",
        "f1_at_budget",
    ],
    "test_split": "pending team confirmation",
    "label_definition": "pending team confirmation",
    "model_results_status": "not yet evaluated",
}

manifest_path = RESULTS_DIR / "evaluation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

print("Saved:", manifest_path)

Saved: /home/ibm-user/shared/results/niyati/evaluation_manifest.json


In [29]:
from pathlib import Path
import os
import sys

print("Current directory:", os.getcwd())
print("Python executable:", sys.executable)
print("Shared project exists:", Path("/home/ibm-user/shared").exists())
print("Package directory:", Path("/home/ibm-user/shared/mulenet").exists())
print("Package initializer:",
      Path("/home/ibm-user/shared/mulenet/__init__.py").exists())

Current directory: /home/ibm-user/shared/notebooks
Python executable: /opt/venv/bin/python3
Shared project exists: True
Package directory: True
Package initializer: True


In [30]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/home/ibm-user/shared").resolve()

if not (PROJECT_ROOT / "mulenet" / "evaluate.py").exists():
    raise FileNotFoundError(
        f"Cannot find mulenet/evaluate.py under {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mulenet.common import load_nodes
from mulenet.evaluate import (
    results_table,
    evaluate_scores,
    uplift_ci,
    chain_detection,
    plot_recall,
)

print("Imports successful.")
print("Project root:", PROJECT_ROOT)


Imports successful.
Project root: /home/ibm-user/shared


In [31]:
from pathlib import Path
import inspect

from mulenet.baselines import run_baseline
from mulenet.common import load_nodes
from mulenet.evaluate import (
    results_table,
    evaluate_scores,
    uplift_ci,
    chain_detection,
    plot_recall,
)

ROOT = Path("/home/ibm-user/shared")
SCORE_DIR = ROOT / "results" / "scores"

print("=== Evaluation function signatures ===")
for fn in [
    results_table,
    evaluate_scores,
    uplift_ci,
    chain_detection,
    plot_recall,
    run_baseline,
]:
    print(f"{fn.__name__}{inspect.signature(fn)}")

print("\n=== Full-data files ===")
for name in [
    "data/processed/nodes.csv",
    "data/processed/edges.npz",
]:
    path = ROOT / name
    print(f"{name}: {'EXISTS' if path.exists() else 'MISSING'}")

print("\n=== Required full-data score files ===")
pairs = {
    "Logistic Regression": ["lr_silo.csv", "lr_combined.csv"],
    "XGBoost": ["xgb_silo.csv", "xgb_combined.csv"],
    "Snap ML graph patterns + XGBoost": [
        "gfpxgb_silo.csv", "gfpxgb_combined.csv"
    ],
    "Graph patterns, own labels only": [
        "gfpxgb_silo.csv", "gfpxgb_combined_ownlabels.csv"
    ],
}

for model, files in pairs.items():
    print(f"\n{model}")
    for filename in files:
        path = SCORE_DIR / filename
        print(f"  {filename}: {'EXISTS' if path.exists() else 'MISSING'}")

print("\n=== Existing final outputs ===")
for name in [
    "results/metrics.csv",
    "results/chains.csv",
    "results/alerts.json",
    "results/figures/recall_at_budget.png",
    "results/figures/uplift_by_member.png",
]:
    path = ROOT / name
    print(f"{name}: {'EXISTS' if path.exists() else 'NOT PRESENT'}")

=== Evaluation function signatures ===
results_table(pairs: 'dict', nodes, chains=None, budget=0.01) -> 'pd.DataFrame'
evaluate_scores(scores, nodes, budget: 'float' = 0.01) -> 'pd.DataFrame'
uplift_ci(silo, combined, nodes, budget=0.01, n_boot=1000, seed=42)
chain_detection(scores, nodes, chains: 'pd.DataFrame', budget=0.01) -> 'dict'
plot_recall(table: 'pd.DataFrame', path: 'str')
run_baseline(kind: 'str', setting: 'str', sample: 'bool' = False, out_dir: 'str | None' = None) -> 'pd.DataFrame'

=== Full-data files ===
data/processed/nodes.csv: EXISTS
data/processed/edges.npz: EXISTS

=== Required full-data score files ===

Logistic Regression
  lr_silo.csv: MISSING
  lr_combined.csv: MISSING

XGBoost
  xgb_silo.csv: MISSING
  xgb_combined.csv: MISSING

Snap ML graph patterns + XGBoost
  gfpxgb_silo.csv: MISSING
  gfpxgb_combined.csv: MISSING

Graph patterns, own labels only
  gfpxgb_silo.csv: MISSING
  gfpxgb_combined_ownlabels.csv: MISSING

=== Existing final outputs ===
results/metr

In [32]:
import inspect
from mulenet.evaluate import results_table, chain_detection, plot_recall

print("=== results_table source ===")
print(inspect.getsource(results_table))

print("\n=== chain_detection source ===")
print(inspect.getsource(chain_detection))

print("\n=== plot_recall source ===")
print(inspect.getsource(plot_recall))

=== results_table source ===
def results_table(pairs: dict, nodes, chains=None, budget=BUDGET) -> pd.DataFrame:
    """pairs = {"GNN": (silo_df, combined_df), "XGBoost": (...), ...} -> the 2x2 headline table."""
    rows = []
    for model, (silo, comb) in pairs.items():
        es, ec = evaluate_scores(silo, nodes, budget), evaluate_scores(comb, nodes, budget)
        u = uplift_ci(silo, comb, nodes, budget)
        row = {"model": model,
               "recall_silo": u["recall_silo"], "recall_combined": u["recall_combined"],
               "uplift_pts": u["uplift_pts"], "uplift_ci_low": u["ci_low"], "uplift_ci_high": u["ci_high"],
               "uplift_relative": u["uplift_pts"] / max(u["recall_silo"], 1e-12),
               "rescued_mules": u["rescued_mules"],
               "pr_auc_silo": es.iloc[-1].pr_auc, "pr_auc_combined": ec.iloc[-1].pr_auc}
        if chains is not None:
            cs, cc = chain_detection(silo, nodes, chains, budget), chain_detection(comb, nodes, chains, b

In [33]:
# ============================================================
# FULL-DATA OFFICIAL EVALUATION — RUN ONLY AFTER TEAM APPROVAL
# Requires validated full-data scores and canonical labels/splits.
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from mulenet.common import load_nodes
from mulenet.evaluate import (
    results_table,
    evaluate_scores,
    uplift_ci,
    chain_detection,
    plot_recall,
)

ROOT = Path("/home/ibm-user/shared")
SCORE_DIR = ROOT / "results" / "scores"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

# Required files. This cell deliberately stops rather than training
# models or silently falling back to sample scores.
score_files = [
    "lr_silo.csv",
    "lr_combined.csv",
    "xgb_silo.csv",
    "xgb_combined.csv",
    "gfpxgb_silo.csv",
    "gfpxgb_combined.csv",
    "gfpxgb_combined_ownlabels.csv",
]

missing = [name for name in score_files if not (SCORE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Full-data evaluation not started. Missing score files:\n"
        + "\n".join(missing)
    )

nodes_path = ROOT / "data" / "processed" / "nodes.csv"
if not nodes_path.is_file():
    raise FileNotFoundError(f"Missing canonical nodes file: {nodes_path}")

nodes = load_nodes(sample=False)

# Load the score files into the official evaluation pairs.
scores = {
    name.removesuffix(".csv"): pd.read_csv(SCORE_DIR / name)
    for name in score_files
}

# Basic safety checks before evaluation.
required_columns = {"acct_key", "inst", "split", "score"}
for name, df in scores.items():
    if not required_columns.issubset(df.columns):
        raise ValueError(f"{name}: missing required score columns")
    if df["acct_key"].duplicated().any():
        raise ValueError(f"{name}: duplicate account IDs")
    if df[list(required_columns)].isna().any().any():
        raise ValueError(f"{name}: missing values in required columns")
    if not pd.to_numeric(df["score"], errors="coerce").notna().all():
        raise ValueError(f"{name}: invalid score values")

# Ensure score account IDs, institution assignments and splits match
# the canonical full dataset before calculating any headline metrics.
canonical = nodes[["acct_key", "inst", "split", "label"]].copy()

for name, df in scores.items():
    check = df.merge(
        canonical,
        on="acct_key",
        how="left",
        suffixes=("_score", "_node"),
        validate="one_to_one",
        indicator=True,
    )

    if not (check["_merge"] == "both").all():
        raise ValueError(f"{name}: score IDs do not match canonical nodes")
    if not (check["inst_score"] == check["inst_node"]).all():
        raise ValueError(f"{name}: institution mismatch")
    if not (check["split_score"] == check["split_node"]).all():
        raise ValueError(f"{name}: split mismatch")

# IMPORTANT:
# Review/confirm the canonical full-data label definition before
# proceeding. This cell verifies ID/institution/split alignment;
# it does not independently establish that labels are authoritative.

pairs = {
    "Logistic Regression": (
        scores["lr_silo"], scores["lr_combined"]
    ),
    "XGBoost": (
        scores["xgb_silo"], scores["xgb_combined"]
    ),
    "Snap ML graph patterns + XGBoost": (
        scores["gfpxgb_silo"], scores["gfpxgb_combined"]
    ),
    "Graph patterns, own labels only": (
        scores["gfpxgb_silo"],
        scores["gfpxgb_combined_ownlabels"],
    ),
}

# Chain file must be confirmed as belonging to this full-data build.
# Set this path only after the team identifies the canonical chain file.
CHAINS_PATH = ROOT / "data" / "processed" / "chains.csv"

if not CHAINS_PATH.is_file():
    raise FileNotFoundError(
        f"Canonical full-data chain file not confirmed/found: {CHAINS_PATH}. "
        "Inspect the repository and set CHAINS_PATH to the verified full-data "
        "chain file before rerunning."
    )

chains = pd.read_csv(CHAINS_PATH)

# Prepare output directories, but do not overwrite existing deliverables.
output_paths = [
    RESULTS_DIR / "metrics.csv",
    RESULTS_DIR / "chains.csv",
    FIGURES_DIR / "recall_at_budget.png",
    FIGURES_DIR / "uplift_by_member.png",
]
existing = [str(p) for p in output_paths if p.exists()]
if existing:
    raise FileExistsError(
        "Existing outputs would be overwritten; inspect them first:\n"
        + "\n".join(existing)
    )

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Official headline comparison.
metrics = results_table(pairs, nodes, chains=chains, budget=0.01)

# Save only after all input validations have passed.
metrics.to_csv(RESULTS_DIR / "metrics.csv", index=False)

# Detailed chain detection by model and setting.
chain_rows = []
for model, (silo_df, combined_df) in pairs.items():
    for setting, score_df in [
        ("silo", silo_df),
        ("combined", combined_df),
    ]:
        result = chain_detection(score_df, nodes, chains, budget=0.01)
        chain_rows.append({
            "model": model,
            "setting": setting,
            **result,
        })

chain_results = pd.DataFrame(chain_rows)
chain_results.to_csv(RESULTS_DIR / "chains.csv", index=False)

# Official recall-at-budget figure.
plot_recall(
    metrics,
    str(FIGURES_DIR / "recall_at_budget.png"),
)

# Per-bank uplift chart for the main GFpxGB comparison.
silo_eval = evaluate_scores(
    scores["gfpxgb_silo"], nodes, budget=0.01
).set_index("inst")
combined_eval = evaluate_scores(
    scores["gfpxgb_combined"], nodes, budget=0.01
).set_index("inst")

bank_names = sorted(
    set(silo_eval.index) & set(combined_eval.index)
)
bank_names = [b for b in bank_names if b != "ALL"]

uplift_pp = [
    (
        combined_eval.loc[b, "recall_at_budget"]
        - silo_eval.loc[b, "recall_at_budget"]
    ) * 100
    for b in bank_names
]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(bank_names, uplift_pp)
ax.axhline(0, linewidth=1)
ax.set_ylabel("Recall uplift (percentage points)")
ax.set_xlabel("Institution")
ax.set_title("GFpxGB combined vs silo by institution")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "uplift_by_member.png", dpi=200)
plt.close(fig)

print("Full-data evaluation completed.")
print(metrics.to_string(index=False))
print("\nCreated:")
for path in output_paths:
    print(path)

FileNotFoundError: Full-data evaluation not started. Missing score files:
lr_silo.csv
lr_combined.csv
xgb_silo.csv
xgb_combined.csv
gfpxgb_silo.csv
gfpxgb_combined.csv
gfpxgb_combined_ownlabels.csv

In [34]:
from pathlib import Path

ROOT = Path("/home/ibm-user/shared")

for folder in [
    ROOT / "results" / "scores",
    ROOT / "results" / "scores_sample",
]:
    print(f"\n=== {folder} ===")
    if not folder.exists():
        print("Directory does not exist")
        continue

    files = sorted(folder.glob("*.csv"))
    if not files:
        print("No CSV files found")
    for path in files:
        print(f"{path.name:40} {path.stat().st_size:>12,} bytes")


=== /home/ibm-user/shared/results/scores ===
No CSV files found

=== /home/ibm-user/shared/results/scores_sample ===
gfpxgb_combined.csv                        18,244,368 bytes
gfpxgb_combined_ownlabels.csv              18,036,239 bytes
gfpxgb_silo.csv                            18,013,085 bytes
gnn_combined.csv                           18,184,804 bytes
gnn_silo.csv                               18,344,606 bytes
gnn_training_log.csv                              558 bytes
lr_combined.csv                            18,112,090 bytes
lr_silo.csv                                18,142,252 bytes
xgb_combined.csv                           18,043,657 bytes
xgb_silo.csv                               18,061,369 bytes


In [35]:
import inspect
from mulenet.evaluate import uplift_ci, alerts

print("=== uplift_ci ===")
print(inspect.getsource(uplift_ci))

print("\n=== alerts ===")
print(inspect.getsource(alerts))

=== uplift_ci ===
def uplift_ci(silo, combined, nodes, budget=BUDGET, n_boot=1000, seed=42):
    """Recall uplift (combined - silo) with a paired 95% bootstrap CI over test mules."""
    a = alerts(silo, nodes, budget).set_index("acct_key")
    b = alerts(combined, nodes, budget).set_index("acct_key")
    mules = a.index[a.label == 1]
    hit_a = a.loc[mules, "alerted"].to_numpy(float)
    hit_b = b.loc[mules, "alerted"].to_numpy(float)
    diff = hit_b.mean() - hit_a.mean()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(mules), size=(n_boot, len(mules)))
    boots = hit_b[idx].mean(1) - hit_a[idx].mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"recall_silo": hit_a.mean(), "recall_combined": hit_b.mean(),
            "uplift_pts": diff, "ci_low": lo, "ci_high": hi,
            "rescued_mules": int(((hit_b == 1) & (hit_a == 0)).sum()),
            "lost_mules": int(((hit_a == 1) & (hit_b == 0)).sum())}


=== alerts ===
def alerts(scores: pd.DataF

In [37]:
from pathlib import Path
import pandas as pd

from mulenet.common import load_nodes
from mulenet.evaluate import uplift_ci

ROOT = Path("/home/ibm-user/shared")
SCORE_DIR = ROOT / "results" / "scores_sample"

silo_path = SCORE_DIR / "gfpxgb_silo.csv"
combined_path = SCORE_DIR / "gfpxgb_combined.csv"

# Check files before loading them
for path in [silo_path, combined_path]:
    print(path, "exists:", path.is_file())

if not silo_path.is_file() or not combined_path.is_file():
    raise FileNotFoundError(
        "One or both sample score files are not present at the expected paths."
    )

nodes = load_nodes(sample=True)
silo = pd.read_csv(silo_path)
combined = pd.read_csv(combined_path)

result = uplift_ci(
    silo,
    combined,
    nodes,
    budget=0.01,
    n_boot=1000,
    seed=42,
)

print("\nOfficial sample GFpxGB uplift result:")
for key, value in result.items():
    if key in {
        "recall_silo", "recall_combined",
        "uplift_pts", "ci_low", "ci_high"
    }:
        print(f"{key}: {value * 100:.3f}%")
    else:
        print(f"{key}: {value}")

/home/ibm-user/shared/results/scores_sample/gfpxgb_silo.csv exists: True
/home/ibm-user/shared/results/scores_sample/gfpxgb_combined.csv exists: True

Official sample GFpxGB uplift result:
recall_silo: 22.024%
recall_combined: 24.107%
uplift_pts: 2.083%
ci_low: -1.786%
ci_high: 5.952%
rescued_mules: 26
lost_mules: 19


In [38]:
from pathlib import Path
import pandas as pd

ROOT = Path("/home/ibm-user/shared")
SCORE_DIR = ROOT / "results" / "scores_sample"

from mulenet.common import load_nodes
from mulenet.evaluate import alerts, evaluate_scores

nodes = load_nodes(sample=True)
silo = pd.read_csv(SCORE_DIR / "gfpxgb_silo.csv")
combined = pd.read_csv(SCORE_DIR / "gfpxgb_combined.csv")

print("Nodes shape:", nodes.shape)
print("Node columns:", nodes.columns.tolist())
print("Silo score rows:", len(silo))
print("Combined score rows:", len(combined))

print("\nNode split counts:")
print(nodes["split"].value_counts(dropna=False))

print("\nNode label counts by split:")
print(pd.crosstab(nodes["split"], nodes["label"]))

print("\nOfficial evaluation tables:")
print("Silo:")
print(evaluate_scores(silo, nodes, budget=0.01).to_string(index=False))
print("\nCombined:")
print(evaluate_scores(combined, nodes, budget=0.01).to_string(index=False))

for name, scores in [("silo", silo), ("combined", combined)]:
    a = alerts(scores, nodes, budget=0.01)
    print(f"\n{name}: alert output")
    print("Rows:", len(a))
    print("Columns:", a.columns.tolist())
    if "split" in a.columns:
        print("Split counts:")
        print(a["split"].value_counts(dropna=False))
    if "label" in a.columns:
        print("Total positives in alert output:", int(a["label"].sum()))
        if "split" in a.columns:
            print(
                "Test positives:",
                int(((a["split"] == "test") & (a["label"] == 1)).sum())
            )
    print("Alerted accounts:", int(a["alerted"].sum()))
    

Nodes shape: (510772, 6)
Node columns: ['node_idx', 'acct_key', 'bank', 'inst', 'label', 'split']
Silo score rows: 510772
Combined score rows: 510772

Node split counts:
split
train    306463
test     102155
val      102154
Name: count, dtype: int64

Node label counts by split:
label       0     1
split              
test   101819   336
train  305455  1008
val    101818   336

Official evaluation tables:
Silo:
inst  test_accounts  mules  alerts  caught  precision_at_budget  recall_at_budget  f1_at_budget   pr_auc  roc_auc
  B1          18275     53     183      12             0.065574          0.226415      0.101695 0.086996 0.826992
  B2          21183     64     212      18             0.084906          0.281250      0.130435 0.070231 0.884555
  B3          20803     86     208      15             0.072115          0.174419      0.102041 0.042983 0.875590
  B4          20739     71     207      15             0.072464          0.211268      0.107914 0.035073 0.877247
  B5          21

In [39]:
from mulenet.evaluate import uplift_ci

result = uplift_ci(
    silo,
    combined,
    nodes,
    budget=0.01,
    n_boot=1000,
    seed=42
)

for key, value in result.items():
    if key in {"recall_silo", "recall_combined", "uplift_pts",
               "ci_low", "ci_high"}:
        print(f"{key}: {value * 100:.3f}%")
    else:
        print(f"{key}: {value}")

recall_silo: 22.024%
recall_combined: 24.107%
uplift_pts: 2.083%
ci_low: -1.786%
ci_high: 5.952%
rescued_mules: 26
lost_mules: 19


In [40]:

from pathlib import Path
import pandas as pd

ROOT = Path("/home/ibm-user/shared")
FULL_SCORES = ROOT / "results" / "scores"
SAMPLE_SCORES = ROOT / "results" / "scores_sample"

expected = [
    "lr_silo.csv",
    "lr_combined.csv",
    "xgb_silo.csv",
    "xgb_combined.csv",
    "gfpxgb_silo.csv",
    "gfpxgb_combined.csv",
    "gfpxgb_combined_ownlabels.csv",
]

print("FULL-DATA SCORE FILES")
for name in expected:
    path = FULL_SCORES / name
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")
    if path.is_file():
        df = pd.read_csv(path)
        print(
            f"  rows={len(df):,}, "
            f"columns={df.columns.tolist()}, "
            f"unique_accounts={df['acct_key'].nunique() if 'acct_key' in df else 'unknown'}"
        )

print("\nPOSSIBLE CHAIN FILES")
for base in [ROOT / "data", ROOT / "results"]:
    if base.exists():
        for path in base.rglob("*chain*.csv"):
            print(path)

print("\nCURRENT OUTPUTS")
for rel in [
    "results/metrics.csv",
    "results/chains.csv",
    "results/alerts.json",
    "results/figures/recall_at_budget.png",
    "results/figures/uplift_by_member.png",
]:
    p = ROOT / rel
    print(f"{rel}: {'EXISTS' if p.exists() else 'not found'}")


FULL-DATA SCORE FILES
lr_silo.csv: MISSING
lr_combined.csv: MISSING
xgb_silo.csv: MISSING
xgb_combined.csv: MISSING
gfpxgb_silo.csv: MISSING
gfpxgb_combined.csv: MISSING
gfpxgb_combined_ownlabels.csv: MISSING

POSSIBLE CHAIN FILES
/home/ibm-user/shared/results/niyati/chains_sample_20261017.csv

CURRENT OUTPUTS
results/metrics.csv: not found
results/chains.csv: not found
results/alerts.json: not found
results/figures/recall_at_budget.png: not found
results/figures/uplift_by_member.png: not found


In [41]:

from pathlib import Path

ROOT = Path("/home/ibm-user/shared")

expected = {
    "lr_silo.csv",
    "lr_combined.csv",
    "xgb_silo.csv",
    "xgb_combined.csv",
    "gfpxgb_silo.csv",
    "gfpxgb_combined.csv",
    "gfpxgb_combined_ownlabels.csv",
}

print("Searching project for expected score files...")
found = {name: [] for name in expected}

for path in ROOT.rglob("*.csv"):
    if path.name in expected:
        found[path.name].append(path)

for name in sorted(expected):
    print(f"\n{name}")
    if found[name]:
        for path in found[name]:
            print(" ", path)
    else:
        print("  NOT FOUND")

print("\nOther score-like files:")
for path in ROOT.rglob("*score*"):
    if path.is_file():
        print(path)


Searching project for expected score files...

gfpxgb_combined.csv
  /home/ibm-user/shared/results/scores_sample/gfpxgb_combined.csv

gfpxgb_combined_ownlabels.csv
  /home/ibm-user/shared/results/scores_sample/gfpxgb_combined_ownlabels.csv

gfpxgb_silo.csv
  /home/ibm-user/shared/results/scores_sample/gfpxgb_silo.csv

lr_combined.csv
  /home/ibm-user/shared/results/scores_sample/lr_combined.csv

lr_silo.csv
  /home/ibm-user/shared/results/scores_sample/lr_silo.csv

xgb_combined.csv
  /home/ibm-user/shared/results/scores_sample/xgb_combined.csv

xgb_silo.csv
  /home/ibm-user/shared/results/scores_sample/xgb_silo.csv

Other score-like files:
/home/ibm-user/shared/.venv-vipul/lib/python3.12/site-packages/jedi/third_party/typeshed/stubs/s2clientprotocol/s2clientprotocol/score_pb2.pyi


In [42]:
from pathlib import Path
import json

ROOT = Path("/home/ibm-user/shared")
NOTEBOOK = ROOT / "notebooks" / "30_eval_niyati.ipynb"

assert NOTEBOOK.is_file(), f"Notebook not found: {NOTEBOOK}"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = ("rescued", "alerts.json", "gnn_silo", "gnn_combined")

matches = 0
for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(word in source.lower() for word in keywords):
        matches += 1
        print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
        print(source)

if matches == 0:
    print("No matching code cells found.")


==================== CELL 8 ====================
from pathlib import Path

for relative_path in [
    "data/processed/nodes.csv",
    "data/processed/edges.npz",
    "mulenet/common.py",
    "results/scores/lr_silo.csv",
    "results/scores/lr_combined.csv",
    "results/scores/gnn_silo.csv",
    "results/scores/gnn_combined.csv",
]:
    path = PROJECT / relative_path
    print(f"{'READY' if path.exists() else 'WAITING'}  {relative_path}")

==================== CELL 24 ====================
from pathlib import Path
import inspect

from mulenet.baselines import run_baseline
from mulenet.common import load_nodes
from mulenet.evaluate import (
    results_table,
    evaluate_scores,
    uplift_ci,
    chain_detection,
    plot_recall,
)

ROOT = Path("/home/ibm-user/shared")
SCORE_DIR = ROOT / "results" / "scores"

print("=== Evaluation function signatures ===")
for fn in [
    results_table,
    evaluate_scores,
    uplift_ci,
    chain_detection,
    plot_recall,
    run_baseline,
]:
   

In [43]:

import json
from pathlib import Path

ROOT = Path("/home/ibm-user/shared")
NOTEBOOK = ROOT / "notebooks" / "30_eval_niyati.ipynb"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "rescued",
    "alerts.json",
    "json.dump",
    "gnn_silo",
    "gnn_combined",
    "combined_score",
    "to_json",
]

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    matched = [term for term in terms if term.lower() in source.lower()]
    if matched:
        print(f"\n{'=' * 18} CELL {i} | MATCHES: {matched} {'=' * 18}")
        print(source)



================== CELL 8 | MATCHES: ['gnn_silo', 'gnn_combined'] ==================
from pathlib import Path

for relative_path in [
    "data/processed/nodes.csv",
    "data/processed/edges.npz",
    "mulenet/common.py",
    "results/scores/lr_silo.csv",
    "results/scores/lr_combined.csv",
    "results/scores/gnn_silo.csv",
    "results/scores/gnn_combined.csv",
]:
    path = PROJECT / relative_path
    print(f"{'READY' if path.exists() else 'WAITING'}  {relative_path}")

================== CELL 21 | MATCHES: ['json.dump'] ==================
manifest = {
    "project": "MuleNet-Z",
    "role": "Role 3 - Baselines and Evaluation",
    "alert_budget": ALERT_BUDGET,
    "alert_budget_scope": "per institution",
    "alert_count_rule": "max(1, round(budget * test_accounts_per_institution))",
    "primary_metric": "recall_at_budget",
    "supporting_metrics": [
        "pr_auc",
        "roc_auc",
        "precision_at_budget",
        "f1_at_budget",
    ],
    "test_split": "pending t

In [44]:

from pathlib import Path

ROOT = Path("/home/ibm-user/shared")

print("=== Existing alerts.json files ===")
for p in ROOT.rglob("alerts.json"):
    if ".venv" not in str(p):
        print(p, f"({p.stat().st_size:,} bytes)")

print("\n=== Possible rescue/export implementations ===")
terms = ("rescued_mules", "rescued_accounts", "json.dump", "gfpxgb_silo")

for base in [ROOT / "mulenet", ROOT / "notebooks", ROOT / "scripts"]:
    if not base.exists():
        continue
    for p in base.rglob("*"):
        if not p.is_file() or p.suffix not in {".py", ".ipynb"}:
            continue
        if ".ipynb_checkpoints" in str(p):
            continue
        try:
            text = p.read_text(encoding="utf-8")
        except (UnicodeDecodeError, OSError):
            continue
        hits = [term for term in terms if term.lower() in text.lower()]
        if hits:
            print(f"\n{p} | matches: {hits}")


=== Existing alerts.json files ===

=== Possible rescue/export implementations ===

/home/ibm-user/shared/mulenet/evaluate.py | matches: ['rescued_mules']

/home/ibm-user/shared/mulenet/gfp.py | matches: ['json.dump', 'gfpxgb_silo']

/home/ibm-user/shared/mulenet/enclave_v1_vipul.py | matches: ['json.dump']

/home/ibm-user/shared/mulenet/enclave.py | matches: ['json.dump']

/home/ibm-user/shared/notebooks/30_eval_niyati.ipynb | matches: ['rescued_mules', 'json.dump', 'gfpxgb_silo']

/home/ibm-user/shared/notebooks/20_gnn_arindam.ipynb | matches: ['json.dump', 'gfpxgb_silo']

/home/ibm-user/shared/notebooks/10_data_vishwajeet.ipynb | matches: ['json.dump']


In [45]:

from pathlib import Path
import json

ROOT = Path("/home/ibm-user/shared")

# Inspect JSON-writing sections in the existing project code.
for rel in [
    "mulenet/enclave_v1_vipul.py",
    "mulenet/enclave.py",
    "mulenet/gfp.py",
]:
    path = ROOT / rel
    print(f"\n{'=' * 20} {rel} {'=' * 20}")
    lines = path.read_text(encoding="utf-8").splitlines()

    hits = [
        i for i, line in enumerate(lines)
        if any(term in line.lower() for term in [
            "json.dump", "alerts.json", "rescued", "alerted",
            "gfpxgb_silo", "gfpxgb_combined"
        ])
    ]

    shown = set()
    for i in hits:
        start, end = max(0, i - 5), min(len(lines), i + 9)
        if any(j in shown for j in range(start, end)):
            continue
        print(f"\n--- lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")
        shown.update(range(start, end))

# Inspect relevant notebook cells.
nb_path = ROOT / "notebooks" / "30_eval_niyati.ipynb"
nb = json.loads(nb_path.read_text(encoding="utf-8"))

print("\n================ NOTEBOOK EXPORT CANDIDATES ================")
for i, cell in enumerate(nb.get("cells", [])):
    src = "".join(cell.get("source", []))
    if cell.get("cell_type") == "code" and any(
        term in src.lower()
        for term in ["rescued_mules", "alerts.json", "json.dump", "gfpxgb_silo"]
    ):
        print(f"\n--- Cell {i} ---\n{src}")




==================== mulenet/enclave_v1_vipul.py ====================

--- lines 103-113 ---
 103:     out = Path(output_dir)
 104:     out.mkdir(parents=True, exist_ok=True)
 105:     written = []
 106: 
 107:     for bank, bank_alerts in grouped.items():
 108:         payload = json.dumps(bank_alerts).encode("utf-8")
 109:         path = out / f"{bank}_alerts.enc"
 110:         path.write_bytes(encrypt_bytes(payload, key))
 111:         written.append(str(path))
 112: 
 113:     return written

==================== mulenet/enclave.py ====================

--- lines 43-56 ---
  43:         return load_keyring(path)
  44:     ring = {"hmac": os.urandom(32), "banks": {b: os.urandom(32) for b in INSTS}}
  45:     path.parent.mkdir(parents=True, exist_ok=True)
  46:     enc = {"hmac": base64.b64encode(ring["hmac"]).decode(),
  47:            "banks": {b: base64.b64encode(k).decode() for b, k in ring["banks"].items()}}
  48:     path.write_text(json.dumps(enc))
  49:     os.chmod(path, 0o

In [46]:

from pathlib import Path

ROOT = Path("/home/ibm-user/shared")

for rel in ["mulenet/enclave_v1_vipul.py", "mulenet/enclave.py"]:
    path = ROOT / rel
    lines = path.read_text(encoding="utf-8").splitlines()

    print(f"\n{'=' * 18} {rel} {'=' * 18}")
    # Show the functions that prepare and write alert payloads.
    for i, line in enumerate(lines):
        if any(term in line.lower() for term in [
            "def bank_submit",
            "def bank_receive",
            "def make_alert",
            "def alerts",
            "def export",
            "bank_alerts",
            "alerted",
        ]):
            start = max(0, i - 3)
            end = min(len(lines), i + 16)
            print(f"\n--- lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")



================== mulenet/enclave_v1_vipul.py ==================

--- lines 53-71 ---
  53:     return AESGCM(key).decrypt(nonce, ciphertext_and_tag, None)
  54: 
  55: 
  56: def write_bank_alerts(
  57:     alerts: list[dict],
  58:     output_dir: str,
  59:     key: bytes,
  60:     hmac_key: bytes,
  61: ) -> list[str]:
  62:     """Write encrypted, bank-specific alerts with pseudonymised account IDs."""
  63:     if not isinstance(key, bytes) or len(key) != 32:
  64:         raise ValueError("AES-256-GCM requires a 32-byte encryption key")
  65:     if not isinstance(hmac_key, bytes) or len(hmac_key) < 32:
  66:         raise ValueError("HMAC key must contain at least 32 bytes")
  67: 
  68:     grouped: dict[str, list[dict]] = {}
  69: 
  70:     for alert in alerts:
  71:         if not isinstance(alert, dict):

--- lines 104-113 ---
 104:     out.mkdir(parents=True, exist_ok=True)
 105:     written = []
 106: 
 107:     for bank, bank_alerts in grouped.items():
 108:        

In [47]:

from pathlib import Path

ROOT = Path("/home/ibm-user/shared")

for rel, start, end in [
    ("mulenet/enclave_v1_vipul.py", 56, 103),
    ("mulenet/enclave.py", 73, 125),
]:
    path = ROOT / rel
    lines = path.read_text(encoding="utf-8").splitlines()

    print(f"\n{'=' * 18} {rel} | lines {start}-{end} {'=' * 18}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")



================== mulenet/enclave_v1_vipul.py | lines 56-103 ==================
  56: def write_bank_alerts(
  57:     alerts: list[dict],
  58:     output_dir: str,
  59:     key: bytes,
  60:     hmac_key: bytes,
  61: ) -> list[str]:
  62:     """Write encrypted, bank-specific alerts with pseudonymised account IDs."""
  63:     if not isinstance(key, bytes) or len(key) != 32:
  64:         raise ValueError("AES-256-GCM requires a 32-byte encryption key")
  65:     if not isinstance(hmac_key, bytes) or len(hmac_key) < 32:
  66:         raise ValueError("HMAC key must contain at least 32 bytes")
  67: 
  68:     grouped: dict[str, list[dict]] = {}
  69: 
  70:     for alert in alerts:
  71:         if not isinstance(alert, dict):
  72:             raise ValueError("Each alert must be a dictionary")
  73: 
  74:         bank = alert.get("bank")
  75:         account_id = alert.get("account_id")
  76: 
  77:         if not isinstance(bank, str) or not re.fullmatch(
  78:             r

In [48]:

from pathlib import Path

ROOT = Path("/home/ibm-user/shared")

for rel in [
    "mulenet/enclave_v1_vipul.py",
    "mulenet/enclave.py",
]:
    path = ROOT / rel
    lines = path.read_text(encoding="utf-8").splitlines()

    print(f"\n{'=' * 16} {rel}: alert generation and demo entry points {'=' * 16}")
    for i, line in enumerate(lines):
        if any(term in line.lower() for term in [
            "def enclave_run",
            "def run_demo",
            "write_bank_alerts(",
            "alerts.json",
            "output_dir",
            "scores_csv",
            "if __name__",
        ]):
            start = max(0, i - 4)
            end = min(len(lines), i + 18)
            print(f"\n--- lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")



================ mulenet/enclave_v1_vipul.py: alert generation and demo entry points ================

--- lines 52-73 ---
  52:     nonce, ciphertext_and_tag = encrypted[:12], encrypted[12:]
  53:     return AESGCM(key).decrypt(nonce, ciphertext_and_tag, None)
  54: 
  55: 
  56: def write_bank_alerts(
  57:     alerts: list[dict],
  58:     output_dir: str,
  59:     key: bytes,
  60:     hmac_key: bytes,
  61: ) -> list[str]:
  62:     """Write encrypted, bank-specific alerts with pseudonymised account IDs."""
  63:     if not isinstance(key, bytes) or len(key) != 32:
  64:         raise ValueError("AES-256-GCM requires a 32-byte encryption key")
  65:     if not isinstance(hmac_key, bytes) or len(hmac_key) < 32:
  66:         raise ValueError("HMAC key must contain at least 32 bytes")
  67: 
  68:     grouped: dict[str, list[dict]] = {}
  69: 
  70:     for alert in alerts:
  71:         if not isinstance(alert, dict):
  72:             raise ValueError("Each alert must be a dicti

In [49]:

from pathlib import Path

path = Path("/home/ibm-user/shared/mulenet/enclave.py")
lines = path.read_text(encoding="utf-8").splitlines()

for i in range(144, min(195, len(lines))):
    print(f"{i + 1:4}: {lines[i]}")


 145:     os.makedirs(out_dir, exist_ok=True)
 146:     written = {}
 147:     for b, g in roster[roster.split == split].groupby("bank"):
 148:         k = max(1, round(budget * len(g)))
 149:         top = g.nlargest(k, "score").set_index("p")
 150:         top = top.join(inflow).join(outflow).join(in_members).fillna(0)
 151:         alerts = [{"account_pseudonym": p, "score": round(float(r.score), 4), "rank": i + 1,
 152:                    "evidence": {"amount_in": round(r.amt_in, 2), "amount_out": round(r.amt_out, 2),
 153:                                 "pass_through": round(r.amt_out / r.amt_in, 3) if r.amt_in else None,
 154:                                 "senders_from_members": int(r.in_members),
 155:                                 "receivers_at_members": int(r.out_members)}}
 156:                   for i, (p, r) in enumerate(top.iterrows())]
 157:         path = Path(out_dir) / f"{b}_alerts.enc"
 158:         path.write_bytes(_seal({"bank": b, "alerts": alerts}, bank_keys

In [50]:

from pathlib import Path
import pandas as pd

from mulenet.common import load_nodes
from mulenet.evaluate import alerts

ROOT = Path("/home/ibm-user/shared")
SCORE_DIR = ROOT / "results" / "scores_sample"

nodes = load_nodes(sample=True)
silo_scores = pd.read_csv(SCORE_DIR / "gfpxgb_silo.csv")
combined_scores = pd.read_csv(SCORE_DIR / "gfpxgb_combined.csv")

# Official alert selection at the frozen 1% per-institution budget.
silo_alerts = alerts(silo_scores, nodes, budget=0.01)
combined_alerts = alerts(combined_scores, nodes, budget=0.01)

# Keep only test accounts and identify rescued mule accounts.
silo_test = silo_alerts[silo_alerts["acct_key"].isin(
    nodes.loc[nodes["split"] == "test", "acct_key"]
)]
combined_test = combined_alerts[combined_alerts["acct_key"].isin(
    nodes.loc[nodes["split"] == "test", "acct_key"]
)]

silo_caught = set(
    silo_test.loc[
        (silo_test["label"] == 1) & (silo_test["alerted"]),
        "acct_key"
    ]
)
combined_caught = set(
    combined_test.loc[
        (combined_test["label"] == 1) & (combined_test["alerted"]),
        "acct_key"
    ]
)

rescued_ids = combined_caught - silo_caught

rescued = combined_test[
    combined_test["acct_key"].isin(rescued_ids)
].copy()

rescued = rescued.sort_values("score", ascending=False).head(20)

print("Silo mules caught:", len(silo_caught))
print("Combined mules caught:", len(combined_caught))
print("Rescued mules:", len(rescued_ids))
print("Top rescued accounts (preview only):")
print(rescued[["acct_key", "inst", "score"]].to_string(index=False))
print("\nNo files written.")


Silo mules caught: 74
Combined mules caught: 94
Rescued mules: 38
Top rescued accounts (preview only):
        acct_key inst    score
  4766_80AF35230   B3 0.972543
 19477_808DEA500   B3 0.967502
 15723_8079F41C0   B3 0.966157
 11852_8017AFC60   B2 0.958918
 19746_807B16D80   B3 0.955160
213952_80DEF52A0   B3 0.953135
    12_8009DED80   B3 0.950486
  2915_80BC96640   B1 0.947861
 11128_80C41D050   B2 0.946239
 25170_8094A1350   B2 0.943965
 48526_8120065B0   B4 0.943209
 29620_80AF61B50   B4 0.940145
     7_805C8AC70   B2 0.932716
 15980_806039280   B1 0.929720
    20_803892E30   B3 0.928426
116843_806EB9F80   B3 0.922361
 35303_80D99E4D0   B4 0.921861
 19535_80B18E250   B4 0.918910
110057_804162D10   B3 0.917232
135403_80E2226E0   B3 0.914430

No files written.


In [51]:

from pathlib import Path
import pandas as pd

from mulenet.common import load_nodes
from mulenet.evaluate import alerts, evaluate_scores

ROOT = Path("/home/ibm-user/shared")
SCORE_DIR = ROOT / "results" / "scores_sample"

nodes = load_nodes(sample=True)
silo_path = SCORE_DIR / "gfpxgb_silo.csv"
combined_path = SCORE_DIR / "gfpxgb_combined.csv"

for p in [silo_path, combined_path]:
    stat = p.stat()
    print(f"{p.name}: modified={pd.Timestamp(stat.st_mtime, unit='s')}, bytes={stat.st_size:,}")

silo = pd.read_csv(silo_path)
combined = pd.read_csv(combined_path)

print("\nOfficial evaluation caught counts:")
for name, scores in [("silo", silo), ("combined", combined)]:
    table = evaluate_scores(scores, nodes, budget=0.01)
    print(f"\n{name}:")
    print(table[["inst", "test_accounts", "mules", "alerts", "caught"]].to_string(index=False))

    a = alerts(scores, nodes, budget=0.01)
    direct = int(((a["label"] == 1) & a["alerted"].astype(bool)).sum())
    print("Direct caught count:", direct)
    print("Alert rows:", len(a), "| alerted:", int(a["alerted"].astype(bool).sum()))

print("\nNo files written.")



gfpxgb_silo.csv: modified=2026-10-17 11:33:29.743270397, bytes=18,013,085
gfpxgb_combined.csv: modified=2026-10-17 11:44:47.093240499, bytes=18,433,957

Official evaluation caught counts:

silo:
inst  test_accounts  mules  alerts  caught
  B1          18275     53     183      12
  B2          21183     64     212      18
  B3          20803     86     208      15
  B4          20739     71     207      15
  B5          21155     62     212      14
 ALL         102155    336    1022      74
Direct caught count: 74
Alert rows: 102155 | alerted: 1022

combined:
inst  test_accounts  mules  alerts  caught
  B1          18275     53     183      15
  B2          21183     64     212      20
  B3          20803     86     208      26
  B4          20739     71     207      19
  B5          21155     62     212      14
 ALL         102155    336    1022      94
Direct caught count: 94
Alert rows: 102155 | alerted: 1022

No files written.


In [52]:

from mulenet.evaluate import uplift_ci

result = uplift_ci(
    silo,
    combined,
    nodes,
    budget=0.01,
    n_boot=1000,
    seed=42,
)

print("Current official result:")
for key, value in result.items():
    if key in {
        "recall_silo", "recall_combined",
        "uplift_pts", "ci_low", "ci_high"
    }:
        print(f"{key}: {value * 100:.3f}%")
    else:
        print(f"{key}: {value}")

print("\nExpected rescued accounts from this score version:",
      result["rescued_mules"])


Current official result:
recall_silo: 22.024%
recall_combined: 27.976%
uplift_pts: 5.952%
ci_low: 1.778%
ci_high: 10.417%
rescued_mules: 38
lost_mules: 18

Expected rescued accounts from this score version: 38


In [53]:

import json

# Uses `rescued` from your earlier preview cell, which is already
# sorted by combined score and limited to the top 20.
assert len(rescued) == 20, f"Expected 20 accounts, got {len(rescued)}"
assert rescued["acct_key"].is_unique
assert rescued["score"].notna().all()

payload = {
    "project": "MuleNet-Z",
    "dataset": "sample",
    "model": "Snap ML graph patterns + XGBoost",
    "alert_budget": 0.01,
    "selection": "Mules alerted by combined but missed by silo; top 20 by combined score",
    "rescued_count": 38,
    "alerts": [
        {
            "acct_key": str(row.acct_key),
            "inst": str(row.inst),
            "score": round(float(row.score), 6),
        }
        for row in rescued.itertuples(index=False)
    ],
}

# Validate JSON serialisation without writing it.
encoded = json.dumps(payload, indent=2, allow_nan=False)
round_trip = json.loads(encoded)

assert len(round_trip["alerts"]) == 20
assert all(
    set(item) == {"acct_key", "inst", "score"}
    for item in round_trip["alerts"]
)

print("Payload validated in memory.")
print("Rescued accounts available:", payload["rescued_count"])
print("Accounts in export:", len(payload["alerts"]))
print("Top score:", payload["alerts"][0]["score"])
print("No files written.")


Payload validated in memory.
Rescued accounts available: 38
Accounts in export: 20
Top score: 0.972543
No files written.


In [54]:

from pathlib import Path

ROOT = Path("/home/ibm-user/shared")
terms = ("alerts.json", "account_id", "acct_key", "account_pseudonym")

search_roots = [
    ROOT / "notebooks",
    ROOT / "mulenet",
    ROOT / "scripts",
]

matches = []
for folder in search_roots:
    if not folder.exists():
        continue
    for path in folder.rglob("*"):
        if (
            not path.is_file()
            or path.suffix not in {".py", ".ipynb", ".md", ".json"}
            or ".ipynb_checkpoints" in str(path)
        ):
            continue
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            continue

        for line_no, line in enumerate(text.splitlines(), start=1):
            if any(term in line for term in terms):
                matches.append((str(path.relative_to(ROOT)), line_no, line.strip()[:220]))

for path, line_no, line in matches[:100]:
    print(f"{path}:{line_no}: {line}")

print(f"\nMatches shown: {min(len(matches), 100)} / {len(matches)}")
print("Read-only search; no files modified.")


notebooks/40_enclave_vipul.ipynb:230: "        \"account_pseudonym\",\n",
notebooks/30_eval_niyati.ipynb:248: "        acct_key, inst, split, score\n",
notebooks/30_eval_niyati.ipynb:251: "        acct_key, inst, split, label\n",
notebooks/30_eval_niyati.ipynb:257: "    required_pred = {\"acct_key\", \"inst\", \"split\", \"score\"}\n",
notebooks/30_eval_niyati.ipynb:258: "    required_labels = {\"acct_key\", \"inst\", \"split\", \"label\"}\n",
notebooks/30_eval_niyati.ipynb:273: "    lab = labels[[\"acct_key\", \"inst\", \"split\", \"label\"]].copy()\n",
notebooks/30_eval_niyati.ipynb:276: "    if pred[\"acct_key\"].duplicated().any():\n",
notebooks/30_eval_niyati.ipynb:279: "    if lab[\"acct_key\"].duplicated().any():\n",
notebooks/30_eval_niyati.ipynb:284: "        on=[\"acct_key\", \"inst\", \"split\"],\n",
notebooks/30_eval_niyati.ipynb:656: "    \"acct_key\": [f\"{bank}_{i}\" for bank in [\"B1\", \"B2\"]\n",
notebooks/30_eval_niyati.ipynb:663: "demo_silo = demo_nodes[[\"acct_key\

In [55]:

import json
from pathlib import Path

path = Path("/home/ibm-user/shared/notebooks/40_enclave_vipul.ipynb")
nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    if any(term in source for term in (
        "alerts.json",
        "account_pseudonym",
        "read_my_alerts",
        "write_bank_alerts",
        "enclave_run",
    )):
        print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
        print(source[:6000])

print("\nRead-only inspection complete; no files modified.")



==================== CELL 2 ====================
import sys
import os
from pathlib import Path
ROOT = Path.home() / 'shared'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pathlib import Path
import os
import time
import pandas as pd

from mulenet.enclave import (
    INSTS,
    make_keyring,
    bank_submit,
    measure_enclave,
    release_keys,
    enclave_run,
    verify_isolation,
    read_my_alerts,
)

ROOT = Path.home() / "shared"
SCORES = ROOT / "results/scores_sample/xgb_combined.csv"
OUT = ROOT / "results/alerts_enc"

assert SCORES.exists(), f"Score file not found: {SCORES}"

print("Project:", ROOT)
print("Score source:", SCORES.name)
print("Banks:", list(INSTS))
print("Sample mode: enabled")
print("Transaction window: 1 day")


==================== CELL 8 ====================
start = time.perf_counter()

result = enclave_run(
    subs,
    keys,
    scores_csv=str(SCORES),
    out_dir=OUT,
    hmac_key=ring["hmac"],
)

enclave_runtime =

In [56]:

import inspect
from pathlib import Path
from mulenet.enclave import enclave_run, read_my_alerts

print("=== enclave_run signature ===")
print(inspect.signature(enclave_run))

print("\n=== read_my_alerts signature ===")
print(inspect.signature(read_my_alerts))

source_path = Path("/home/ibm-user/shared/mulenet/enclave.py")
source = source_path.read_text(encoding="utf-8").splitlines()

print("\n=== Relevant exporter and output code ===")
for start, end in [(120, 180), (180, 240)]:
    print(f"\n--- Lines {start}-{end} ---")
    for line_no in range(start, min(end, len(source)) + 1):
        print(f"{line_no}: {source[line_no - 1]}")

print("\nRead-only inspection complete; no files modified.")


ModuleNotFoundError: No module named 'cryptography'

In [57]:

import ast
from pathlib import Path

ROOT = Path.home() / "shared"
source_path = ROOT / "mulenet" / "enclave.py"

source = source_path.read_text(encoding="utf-8")
tree = ast.parse(source)

print("=== Function signatures (source inspection only) ===")
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        if node.name in {"enclave_run", "read_my_alerts"}:
            print(f"\n{node.name}, line {node.lineno}")
            print(ast.get_source_segment(source, node))

print("\n=== Output-related source lines ===")
for line_no, line in enumerate(source.splitlines(), start=1):
    if any(term in line for term in (
        "alerts.enc", "json.dump", "json.dumps",
        "account_pseudonym", "write_text", "to_csv",
    )):
        print(f"{line_no}: {line.strip()}")

print("\nRead-only source inspection complete.")


=== Function signatures (source inspection only) ===

enclave_run, line 120
def enclave_run(submissions: dict, bank_keys: dict, scores_csv: str, out_dir: Path,
                hmac_key: bytes, budget: float = BUDGET, split: str = "test") -> dict:
    """Decrypt, join into one graph, score, and write one encrypted alert file per bank.

    Scoring: the joined graph here is exactly the 'combined' view, so we use the
    combined model's scores (scores_csv, keyed by pseudonym inside the enclave).
    """
    t0 = time.perf_counter()
    subs = {b: _open(blob, bank_keys[b], f"submission:{b}") for b, blob in submissions.items()}
    t_dec = time.perf_counter() - t0

    tx = pd.concat([pd.DataFrame(s["tx"]) for s in subs.values()], ignore_index=True)
    roster = pd.concat([pd.DataFrame({**s["roster"], "bank": b}) for b, s in subs.items()], ignore_index=True)

    sc = pd.read_csv(scores_csv, dtype={"acct_key": str})
    sc["p"] = [pseudonym(k, hmac_key) for k in sc["acct_key"]]   # model o

In [58]:

# Read-only preview of the already validated in-memory payload.
# Does not write files or modify Vipul's enclave workflow.

assert payload["dataset"] == "sample"
assert payload["model"] == "Snap ML graph patterns + XGBoost"
assert len(payload["alerts"]) == 20
assert payload["rescued_count"] == 38

print("Proposed export:", "results/alerts.json")
print("Dataset:", payload["dataset"])
print("Model:", payload["model"])
print("Alert budget:", payload["alert_budget"])
print("Total rescued mules:", payload["rescued_count"])
print("Accounts selected for preview:", len(payload["alerts"]))
print("Fields per record:", list(payload["alerts"][0].keys()))
print("Top score:", payload["alerts"][0]["score"])
print("File write performed: No")


Proposed export: results/alerts.json
Dataset: sample
Model: Snap ML graph patterns + XGBoost
Alert budget: 0.01
Total rescued mules: 38
Accounts selected for preview: 20
Fields per record: ['acct_key', 'inst', 'score']
Top score: 0.972543
File write performed: No


In [59]:

import json
from pathlib import Path

ROOT = Path.home() / "shared"
OUT = ROOT / "results" / "alerts.json"

# Confirm the validated payload is still in memory.
assert "payload" in globals(), "Payload is missing; rebuild and validate it first."
assert payload["dataset"] == "sample"
assert payload["model"] == "Snap ML graph patterns + XGBoost"
assert payload["alert_budget"] == 0.01
assert payload["rescued_count"] == 38
assert len(payload["alerts"]) == 20
assert len({a["acct_key"] for a in payload["alerts"]}) == 20

# Do not overwrite any existing shared output.
assert not OUT.exists(), (
    f"{OUT} already exists. Inspect it before deciding whether to replace it."
)

# Validate serialization before writing.
encoded = json.dumps(payload, indent=2, allow_nan=False)
round_trip = json.loads(encoded)
assert len(round_trip["alerts"]) == 20

OUT.parent.mkdir(parents=True, exist_ok=True)
with OUT.open("x", encoding="utf-8") as f:
    f.write(encoded + "\n")

# Verify the saved file.
saved = json.loads(OUT.read_text(encoding="utf-8"))
assert saved == round_trip

print("Export written and verified:", OUT)
print("Dataset:", saved["dataset"])
print("Model:", saved["model"])
print("Rescued mules:", saved["rescued_count"])
print("Accounts exported:", len(saved["alerts"]))
print("Existing files overwritten: No")


Export written and verified: /home/ibm-user/shared/results/alerts.json
Dataset: sample
Model: Snap ML graph patterns + XGBoost
Rescued mules: 38
Accounts exported: 20
Existing files overwritten: No


In [60]:

from pathlib import Path

ROOT = Path.home() / "shared"

required_scores = [
    "lr_silo.csv",
    "lr_combined.csv",
    "xgb_silo.csv",
    "xgb_combined.csv",
    "gfpxgb_silo.csv",
    "gfpxgb_combined.csv",
    "gfpxgb_combined_ownlabels.csv",
]

print("=== Full-data score status ===")
score_dir = ROOT / "results" / "scores"

for name in required_scores:
    p = score_dir / name
    print(
        f"{'FOUND' if p.is_file() else 'MISSING':7} {name}"
        + (f" ({p.stat().st_size:,} bytes)" if p.is_file() else "")
    )

print("\n=== Canonical data files ===")
for rel in [
    "data/processed/nodes.csv",
    "data/processed/edges.npz",
    "data/processed/chains.csv",
]:
    p = ROOT / rel
    print(f"{'FOUND' if p.is_file() else 'MISSING':7} {rel}")

print("\n=== My evaluation notebook ===")
print(ROOT / "notebooks" / "30_eval_niyati.ipynb")
print("Read-only check complete; no files modified.")


=== Full-data score status ===
MISSING lr_silo.csv
MISSING lr_combined.csv
MISSING xgb_silo.csv
MISSING xgb_combined.csv
MISSING gfpxgb_silo.csv
MISSING gfpxgb_combined.csv
MISSING gfpxgb_combined_ownlabels.csv

=== Canonical data files ===
FOUND   data/processed/nodes.csv
FOUND   data/processed/edges.npz
MISSING data/processed/chains.csv

=== My evaluation notebook ===
/home/ibm-user/shared/notebooks/30_eval_niyati.ipynb
Read-only check complete; no files modified.


In [61]:

import ast
from pathlib import Path

ROOT = Path.home() / "shared"
terms = ("laundering_chains", "chain_detection")

for path in ROOT.rglob("*.py"):
    if any(part in {".git", ".venv", "__pycache__"} for part in path.parts):
        continue
    try:
        source = path.read_text(encoding="utf-8")
        tree = ast.parse(source)
    except (OSError, SyntaxError, UnicodeDecodeError):
        continue

    hits = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name in terms:
                hits.append(
                    (node.lineno, ast.get_source_segment(source, node))
                )
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id in terms:
                hits.append((node.lineno, None))

    if hits:
        print(f"\n=== {path.relative_to(ROOT)} ===")
        for line_no, code in hits:
            print(f"\nLine {line_no}")
            print(code if code else "Function call found")

print("\nRead-only search complete.")



=== mulenet/evaluate.py ===

Line 73
def chain_detection(scores, nodes, chains: pd.DataFrame, budget=BUDGET) -> dict:
    """Definition 9: a chain is detected if any of its test accounts is alerted."""
    al = alerts(scores, nodes, budget)[["acct_key", "alerted"]]
    c = chains[chains.split == "test"].merge(al, on="acct_key")
    per = c.groupby("chain_id").agg(members=("chain_members", "first"),
                                    detected=("alerted", "max"),
                                    share_alerted=("alerted", "mean"))
    cross = per[per.members >= 2]
    return {"chains_with_test_accounts": len(per), "chains_detected": int(per.detected.sum()),
            "cross_bank_chains": len(cross), "cross_bank_detected": int(cross.detected.sum()),
            "mean_share_alerted": float(per.share_alerted.mean())}

Line 99
Function call found

Line 99
Function call found

=== mulenet/common.py ===

Line 157
def laundering_chains(sample: bool = False) -> pd.DataFrame:
    """One row

In [62]:

from pathlib import Path
import sys

ROOT = Path.home() / "shared"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mulenet.common import laundering_chains

chains = laundering_chains(sample=False)

required = {
    "node_idx", "acct_key", "inst", "split",
    "chain_id", "chain_size", "chain_members",
}

print("Rows:", len(chains))
print("Unique accounts:", chains["acct_key"].nunique())
print("Duplicate account keys:", int(chains["acct_key"].duplicated().sum()))
print("Missing required columns:", sorted(required - set(chains.columns)))
print("\nSplit counts:")
print(chains["split"].value_counts(dropna=False).to_string())
print("\nChain-member counts:")
print(chains["chain_members"].value_counts().sort_index().to_string())

assert required.issubset(chains.columns)
assert not chains["acct_key"].duplicated().any()
assert chains["split"].isin(["train", "val", "test"]).all()

print("\nFull-data chains validated in memory; no files written.")


Rows: 6357
Unique accounts: 6357
Duplicate account keys: 0
Missing required columns: []

Split counts:
split
train    3814
test     1272
val      1271

Chain-member counts:
chain_members
1     657
2    2618
3     498
4     512
5    2072

Full-data chains validated in memory; no files written.


In [63]:

import pandas as pd
from pathlib import Path

ROOT = Path.home() / "shared"
nodes_path = ROOT / "data" / "processed" / "nodes.csv"

nodes = pd.read_csv(
    nodes_path,
    usecols=["acct_key", "inst", "split"],
    dtype={"acct_key": str},
)
canonical = nodes.drop_duplicates("acct_key")

assert canonical["acct_key"].is_unique, (
    "Canonical nodes contain duplicate account keys; investigate before proceeding."
)

chain_keys = set(chains["acct_key"].astype(str))
node_keys = set(canonical["acct_key"].astype(str))

merged = chains.assign(acct_key=chains["acct_key"].astype(str)).merge(
    canonical.assign(acct_key=canonical["acct_key"].astype(str)),
    on="acct_key",
    how="left",
    suffixes=("_chain", "_node"),
    indicator=True,
    validate="one_to_one",
)

print("Canonical node accounts:", len(canonical))
print("Chain accounts:", len(chains))
print("Chain accounts absent from canonical nodes:", len(chain_keys - node_keys))
print(
    "Institution mismatches:",
    int((merged["inst_chain"] != merged["inst_node"]).sum()),
)
print(
    "Split mismatches:",
    int((merged["split_chain"] != merged["split_node"]).sum()),
)
print("Unmatched rows:", int((merged["_merge"] != "both").sum()))

assert (merged["_merge"] == "both").all(), "Some chain accounts are unmatched."
assert (merged["inst_chain"] == merged["inst_node"]).all(), "Institution mismatch."
assert (merged["split_chain"] == merged["split_node"]).all(), "Split mismatch."

print("\nChain-to-node alignment passed. No files written.")


Canonical node accounts: 515088
Chain accounts: 6357
Chain accounts absent from canonical nodes: 0
Institution mismatches: 0
Split mismatches: 0
Unmatched rows: 0

Chain-to-node alignment passed. No files written.


In [1]:

import subprocess
from pathlib import Path

ROOT = Path.home() / "shared"

for args in [
    ["git", "status", "--short"],
    ["git", "branch", "--show-current"],
    ["git", "log", "-1", "--oneline"],
]:
    result = subprocess.run(
        args,
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print("$", " ".join(args))
    print(result.stdout or result.stderr)


$ git status --short
 M mulenet/common.py
 M notebooks/40_enclave_vipul.ipynb
?? mulenet/__init__.py
?? mulenet/baselines.py
?? mulenet/enclave.py.preprivacy
?? mulenet/enclave_v1_vipul.py
?? mulenet/evaluate.py
?? mulenet/gfp.py
?? mulenet/gnn.py
?? mulenet/privacy_adapter_v1.py
?? mulenet/toy.py
?? notebooks/01_setup_check_arindam.ipynb
?? notebooks/20_gnn_arindam.ipynb
?? notebooks/30_eval_niyati.ipynb
?? notebooks/40_enclave_vipul_executed.ipynb
?? results/

$ git branch --show-current
main

$ git log -1 --oneline
c0f223d Update Vishwajeet data notebook



In [2]:

import subprocess
from pathlib import Path

ROOT = Path.home() / "shared"

for relative in [
    "notebooks/30_eval_niyati.ipynb",
    "results/alerts.json",
    "results/metrics.csv",
    "results/chains.csv",
    "results/figures",
    "results/niyati",
]:
    path = ROOT / relative
    print(f"\n{relative}:")
    if path.is_file():
        print(f"  FILE | {path.stat().st_size:,} bytes")
    elif path.is_dir():
        for item in sorted(path.rglob("*")):
            if item.is_file():
                print(f"  {item.relative_to(ROOT)} | {item.stat().st_size:,} bytes")
    else:
        print("  NOT PRESENT")

print("\nNo Git changes made.")



notebooks/30_eval_niyati.ipynb:
  FILE | 208,009 bytes

results/alerts.json:
  FILE | 2,121 bytes

results/metrics.csv:
  NOT PRESENT

results/chains.csv:
  NOT PRESENT

results/figures:

results/niyati:
  results/niyati/alerts_sample_20261017.json | 27,055 bytes
  results/niyati/chains_sample_20261017.csv | 64,807 bytes
  results/niyati/evaluation_manifest.json | 510 bytes
  results/niyati/metrics_all_models_sample_20261017.csv | 757 bytes
  results/niyati/metrics_baselines_sample_20261017.csv | 584 bytes
  results/niyati/recall_at_budget_sample_20261017.png | 52,511 bytes

No Git changes made.
